Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/openlanguagemodel/openlanguagemodel.git to C:\Users\Dell\AppData\Local\Temp\pip-req-build-qd5ubarc
  Resolved https://github.com/openlanguagemodel/openlanguagemodel.git to commit eeab195fcd3447fd5d6c39b59ca9224fb66b3ac7
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
INFO: pip is looking at multiple versions of openlanguagemodel to determine which version is compatible with other requirements. This could take a while.


  Running command git clone --filter=blob:none --quiet https://github.com/openlanguagemodel/openlanguagemodel.git 'C:\Users\Dell\AppData\Local\Temp\pip-req-build-qd5ubarc'
ERROR: Package 'openlanguagemodel' requires a different Python: 3.14.2 not in '<3.13,>=3.10'


In [5]:
import sys
!{sys.executable} -m pip install git+https://github.com/openlanguagemodel/openlanguagemodel.git

  Cloning https://github.com/openlanguagemodel/openlanguagemodel.git to c:\users\dell\appdata\local\temp\pip-req-build-gab0thoc
  Resolved https://github.com/openlanguagemodel/openlanguagemodel.git to commit eeab195fcd3447fd5d6c39b59ca9224fb66b3ac7
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Obtaining dependency information for torch>=2.1.0 from https://files.pythonhosted.org/packages/be/16/9489b137112040f9911d7527e452854f21cc4e499ce0da79864e6a7451a7/torch-2.14.0-cp312-cp312-win_amd64.whl.metadata
  Using cached torch-2.14.0-cp312-cp312-win_amd64.whl.metadata (38 kB)
  Obtaining dependency information for numpy>=1.20.0 from https://files.pythonhosted.org/packages/7f/b9/87fea2769fe1c47c1b5b01d8310772c9d1

  Running command git clone --filter=blob:none --quiet https://github.com/openlanguagemodel/openlanguagemodel.git 'C:\Users\Dell\AppData\Local\Temp\pip-req-build-gab0thoc'

[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import olm
print("OLM:", olm.__file__)
import math
import random
import time
import os

import torch

from transformers import Qwen2Config, Qwen2ForCausalLM, AutoTokenizer
from olm.data.tokenization import HFTokenizer

tokenizer = HFTokenizer("Qwen/Qwen2.5-0.5B")

from olm.data.datasets import DataLoader, FineWebEduDataset
from olm.train import AutoTrainer
from olm.train.optim import AdamW

OLM: c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\olm\__init__.py


In [7]:


CONTEXT_LENGTH = 1024

HIDDEN_SIZE = 896
INTERMEDIATE_SIZE = 4864
NUM_HIDDEN_LAYERS = 24
NUM_ATTENTION_HEADS = 14
NUM_KEY_VALUE_HEADS = 2


RMS_NORM_EPS = 1e-6
ROPE_THETA = 1000000.0

MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 32

MAX_STEPS = 20

TARGET_TOKENS = 1_000_000_000

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected.")

device: cpu


In [12]:
from olm.data.tokenization import HFTokenizer

tokenizer = HFTokenizer("Qwen/Qwen2.5-0.5B")

print("vocab size:", tokenizer.vocab_size)


vocab size: 151643


In [13]:
effective_batch = (
    MICRO_BATCH_SIZE
    * GRAD_ACCUM_STEPS
)

tokens_per_step = (
    effective_batch
    * CONTEXT_LENGTH
)

target_steps = math.ceil(
    TARGET_TOKENS / tokens_per_step
)

print("micro batch:", MICRO_BATCH_SIZE)
print("gradient accumulation:", GRAD_ACCUM_STEPS)
print("effective batch:", effective_batch)
print("context length:", CONTEXT_LENGTH)
print()
print("tokens / optimizer step:", f"{tokens_per_step:,}")
print("target tokens:", f"{TARGET_TOKENS:,}")
print("target steps:", f"{target_steps:,}")

micro batch: 1
gradient accumulation: 32
effective batch: 32
context length: 1024

tokens / optimizer step: 32,768
target tokens: 1,000,000,000
target steps: 30,518


In [18]:
from olm.data.datasets import FineWebEduDataset, DataLoader

dataset = FineWebEduDataset(
    tokenizer=tokenizer,
    subset="sample-10BT",
    split="train",
    context_length=1024,
    streaming=True,
    shuffle=True,
    seed=42,
)

loader = DataLoader(
    dataset,
    batch_size=4,
    num_workers=0,
)



In [19]:
x, y = next(iter(loader))

print("input shape:", tuple(x.shape))
print("target shape:", tuple(y.shape))

try:
    print(tokenizer.decode(x[0][:200]))
except Exception:
    pass

c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


input shape: (4, 1024)
target shape: (4, 1024)
a combination of one or more elementary reaction steps which start with the appropriate reactants and end with the appropriate product(s)
a description of the path, or sequence of steps, by which a reaction occurs
a description of the path that a reaction takes
a detailed description of how a chemical reaction occurs
a detailed description of the way a reaction occurs and is based on the known experimental data about the reaction
a detailed (theoretical) description of how we think the chemical reaction proceeds
a series of elementary reactions or elementary steps that lead from reactants to products
a set of steps at the molecular level
a step by step description of the separate steps that occur during a chemical reaction
a stepwise description of the reaction path
mechanism. A list of all elementary reactions that occur in the course of an overall chemical reaction.
In chemistry, a reaction mechanism is the step by step sequence of eleme

In [20]:
config = Qwen2Config(
    vocab_size=tokenizer.vocab_size,

    hidden_size=HIDDEN_SIZE,
    intermediate_size=INTERMEDIATE_SIZE,

    num_hidden_layers=NUM_HIDDEN_LAYERS,

    num_attention_heads=NUM_ATTENTION_HEADS,
    num_key_value_heads=NUM_KEY_VALUE_HEADS,

    max_position_embeddings=CONTEXT_LENGTH,

    rms_norm_eps=RMS_NORM_EPS,
    rope_theta=ROPE_THETA,

    hidden_act="silu",

    attention_dropout=0.0,

    use_cache=False,

    tie_word_embeddings=True,
)

model = Qwen2ForCausalLM(config)

model = model.to(device)

params = sum(
    p.numel()
    for p in model.parameters()
)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("parameters:", f"{params:,}")
print("parameters (B):", f"{params / 1e9:.3f}B")
print("trainable:", f"{trainable:,}")
print(
    "approx FP32 model size:",
    f"{params * 4 / 1e9:.2f} GB"
)

parameters: 493,770,240
parameters (B): 0.494B
trainable: 493,770,240
approx FP32 model size: 1.98 GB


In [21]:


model.eval()

with torch.no_grad():

    test_x = x.to(device)

    outputs = model(
        input_ids=test_x
    )

print("input:", tuple(test_x.shape))
print("logits:", tuple(outputs.logits.shape))

model.train()

print("Forward pass successful.")

input: (4, 1024)
logits: (4, 1024, 151643)
Forward pass successful.


In [23]:


trainer = AutoTrainer(
    model,
    AdamW,
    loader,

    device="auto",

    context_length=CONTEXT_LENGTH,

    learning_rate=3e-4,

    weight_decay=0.1,

    grad_accum_steps=GRAD_ACCUM_STEPS,

    use_amp=True,

    grad_clip_norm=1.0,

    preset="balanced",

    verbose=True,
)

print("Trainer created.")

Device Detection Results:
----------------------------------------------------------------------
  CUDA Available: False
  Number of GPUs: 0
  Number of CPUs: 14
  Distributed Training: No

Training Strategy Selection:
----------------------------------------------------------------------
  Strategy: single_cpu
  Device Type: cpu
  Number of GPUs: 0
  Backend: gloo


Model Memory Estimation:
----------------------------------------------------------------------
  Total Parameters: 493,770,240 (493.77M)
  Trainable Parameters: 493,770,240 (493.77M)
  Parameter Memory: 1.84 GB
  Gradient Memory: 1.84 GB
  Optimizer Memory: 3.68 GB
  Estimated Activation Memory: 0.37 GB
  Total Training Memory: 7.73 GB

Initializing single-device Trainer (device: cpu)
Trainer created.


In [24]:


if device == "cpu":

    raise RuntimeError(
        "GPU not detected. "
        "Switch Colab to a GPU runtime."
    )

torch.cuda.empty_cache()

torch.cuda.synchronize()

start_time = time.time()

losses = trainer.train(
    epochs=1,
    max_steps=MAX_STEPS,
    log_interval=5,
)

torch.cuda.synchronize()

elapsed = time.time() - start_time

seconds_per_step = elapsed / MAX_STEPS

steps_per_second = (
    MAX_STEPS / elapsed
)

tokens_per_second = (
    MAX_STEPS * tokens_per_step / elapsed
)

estimated_seconds = (
    target_steps * seconds_per_step
)

estimated_hours = (
    estimated_seconds / 3600
)

estimated_days = (
    estimated_hours / 24
)

print()
print("=" * 70)
print("PREFLIGHT RESULTS")
print("=" * 70)

print(
    "parameters:",
    f"{params / 1e9:.3f}B"
)

print(
    "steps completed:",
    MAX_STEPS
)

print(
    "tokens processed:",
    f"{trainer.total_tokens_processed:,}"
)

print(
    "final loss:",
    losses[-1]
)

print()

print(
    "seconds / optimizer step:",
    f"{seconds_per_step:.2f}"
)

print(
    "steps / second:",
    f"{steps_per_second:.4f}"
)

print(
    "tokens / second:",
    f"{tokens_per_second:,.0f}"
)

print()

print("=" * 70)
print("1B TOKEN TRAINING ESTIMATE")
print("=" * 70)

print(
    "target tokens:",
    f"{TARGET_TOKENS:,}"
)

print(
    "target optimizer steps:",
    f"{target_steps:,}"
)

print(
    "estimated hours:",
    f"{estimated_hours:.2f}"
)

print(
    "estimated days:",
    f"{estimated_days:.2f}"
)

print("=" * 70)

RuntimeError: GPU not detected. Switch Colab to a GPU runtime.

In [ ]:


checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": trainer.optimizer.state_dict(),

    "config": config.to_dict(),

    "losses": losses,

    "context_length": CONTEXT_LENGTH,

    "micro_batch_size": MICRO_BATCH_SIZE,

    "grad_accum_steps": GRAD_ACCUM_STEPS,

    "max_steps": MAX_STEPS,

    "target_tokens": TARGET_TOKENS,

    "target_steps": target_steps,

    "tokens_per_step": tokens_per_step,

    "tokens_per_second": tokens_per_second,

    "estimated_hours": estimated_hours,
}

checkpoint_path = (
    "/content/qwen25_style_0.5b_preflight.pt"
)

torch.save(
    checkpoint,
    checkpoint_path
)

print("saved:", checkpoint_path)

In [ ]:


FINAL_STEPS = target_steps

print("Starting full pretraining")
print()
print("Model:", f"{params / 1e9:.3f}B parameters")
print("Target tokens:", f"{TARGET_TOKENS:,}")
print("Target steps:", f"{FINAL_STEPS:,}")
print()

torch.cuda.empty_cache()

torch.cuda.synchronize()

full_start = time.time()

losses = trainer.train(
    epochs=1,
    max_steps=FINAL_STEPS,
    log_interval=100,
)

torch.cuda.synchronize()

full_elapsed = time.time() - full_start

print()
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    "training time:",
    f"{full_elapsed / 3600:.2f} hours"
)

print(
    "tokens processed:",
    f"{trainer.total_tokens_processed:,}"
)

print(
    "final loss:",
    losses[-1]
)

print("=" * 70)

In [ ]:


FINAL_PATH = "/content/qwen25_style_0.5b_fineweb_edu"

os.makedirs(
    FINAL_PATH,
    exist_ok=True
)

model.save_pretrained(FINAL_PATH)

tokenizer.save_pretrained(FINAL_PATH)

torch.save(
    {
        "losses": losses,
        "tokens_processed": trainer.total_tokens_processed,
        "training_time_seconds": full_elapsed,
    },
    os.path.join(
        FINAL_PATH,
        "training_info.pt"
    )
)

print("Saved model to:")
print(FINAL_PATH)